In [3]:
!pip uninstall -y great_expectations
!pip install "great_expectations==0.18.19"

Found existing installation: great_expectations 1.9.1
Uninstalling great_expectations-1.9.1:
ERROR: Exception:
Traceback (most recent call last):
  File "/usr/lib/python3.10/shutil.py", line 816, in move
    os.rename(src, real_dst)
PermissionError: [Errno 13] Permission denied: '/usr/local/lib/python3.10/dist-packages/great_expectations-1.9.1.dist-info/' -> '/tmp/pip-uninstall-km9k6c5y'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/pip/_internal/cli/base_command.py", line 107, in _run_wrapper
    status = _inner_run()
  File "/usr/local/lib/python3.10/dist-packages/pip/_internal/cli/base_command.py", line 98, in _inner_run
    return self.run(options, args)
  File "/usr/local/lib/python3.10/dist-packages/pip/_internal/commands/uninstall.py", line 105, in run
    uninstall_pathset = req.uninstall(
  File "/usr/local/lib/python3.10/dist-packages/pip/_internal/req/req_install.py", l

In [6]:
import great_expectations as gx
from great_expectations.dataset import SparkDFDataset
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, regexp_replace
import pandas as pd
from IPython.display import display, HTML

HDFS_HOST = "hdfs://spark-master:9000"
PATH_LISTINGS = f"{HDFS_HOST}/data/airbnb/Listings.csv"
PATH_REVIEWS = f"{HDFS_HOST}/data/airbnb/Reviews.csv"
VALID_ROOMS = ["Entire home/apt", "Private room", "Shared room", "Hotel room"]

audit_log = []

def run_check(dataset_name, rule_name, result):
    status = "PASS" if result.success else "FAIL"
    unexpected_count = result.result.get("unexpected_count", 0)
    unexpected_percent = result.result.get("unexpected_percent", 0.0)
    
    audit_log.append({
        "Dataset": dataset_name,
        "Regla de Negocio": rule_name,
        "Estado": status,
        "Filas Erradas": unexpected_count,
        "% Error": unexpected_percent
    })

def render_dashboard():
    df_results = pd.DataFrame(audit_log)
    
    total_rules = len(df_results)
    passed_rules = len(df_results[df_results["Estado"] == "PASS"])
    score = (passed_rules / total_rules) * 100
    
    summary_html = f"""
    <div style="background-color: #f4f4f4; padding: 20px; border-radius: 10px; border-left: 5px solid #007acc; font-family: sans-serif;">
        <h2 style="color: #333; margin-top:0;">Dashboard de Calidad del Dato (Great Expectations)</h2>
        <div style="display: flex; justify-content: space-around; text-align: center;">
            <div>
                <h3 style="margin:0; font-size: 30px; color: #333;">{total_rules}</h3>
                <span style="color: #666; font-size: 14px;">Reglas Ejecutadas</span>
            </div>
            <div>
                <h3 style="margin:0; font-size: 30px; color: {'green' if score == 100 else 'orange'};">{score:.1f}%</h3>
                <span style="color: #666; font-size: 14px;">Índice de Calidad</span>
            </div>
            <div>
                <h3 style="margin:0; font-size: 30px; color: {'red' if total_rules - passed_rules > 0 else 'green'};">{total_rules - passed_rules}</h3>
                <span style="color: #666; font-size: 14px;">Alertas Críticas</span>
            </div>
        </div>
    </div>
    <br>
    """
    display(HTML(summary_html))
    
    def color_status(val):
        color = '#d4edda' if val == 'PASS' else '#f8d7da'
        text_color = '#155724' if val == 'PASS' else '#721c24'
        return f'background-color: {color}; color: {text_color}; font-weight: bold;'

    styled_table = df_results.style \
        .map(color_status, subset=['Estado']) \
        .bar(subset=['% Error'], color='#ffcccc', vmin=0, vmax=100) \
        .format({'% Error': "{:.2f}%", 'Filas Erradas': "{:,}"}) \
        .set_properties(**{'text-align': 'left', 'padding': '10px'}) \
        .set_table_styles([
            {'selector': 'th', 'props': [('background-color', '#007acc'), ('color', 'white'), ('font-size', '14px')]}
        ]) \
        .hide(axis="index")

    display(styled_table)

def main():
    spark = SparkSession.builder \
        .appName("Enterprise_DQ_Dashboard") \
        .master("spark://spark-master:7077") \
        .config("spark.executor.memory", "1g") \
        .getOrCreate()
    
    spark.sparkContext.setLogLevel("ERROR")

    df_listings = spark.read.csv(PATH_LISTINGS, header=True, inferSchema=True, quote="\"", escape="\"", multiLine=True)
    df_listings = df_listings.withColumn("price_clean", regexp_replace(col("price"), r"[\$,]", "").cast("double"))
    gx_listings = SparkDFDataset(df_listings)

    run_check("LISTINGS", "Integridad: Listing_ID Único", gx_listings.expect_column_values_to_be_unique("listing_id"))
    run_check("LISTINGS", "Consistencia: Precio > 0", gx_listings.expect_column_values_to_be_between("price_clean", min_value=0.01))
    run_check("LISTINGS", "Validez: Taxonomía de Habitación", gx_listings.expect_column_values_to_be_in_set("room_type", VALID_ROOMS))
    run_check("LISTINGS", "Completitud: Metadatos Host", gx_listings.expect_column_values_to_not_be_null("host_since"))

    df_reviews = spark.read.csv(PATH_REVIEWS, header=True, inferSchema=True, quote="\"", escape="\"", multiLine=True)
    gx_reviews = SparkDFDataset(df_reviews)

    run_check("REVIEWS", "Integridad: Listing_ID existe (FK)", gx_reviews.expect_column_values_to_not_be_null("listing_id"))
    run_check("REVIEWS", "Validez: Formato Fecha ISO", gx_reviews.expect_column_values_to_match_regex("date", r"\d{4}-\d{2}-\d{2}"))
    run_check("REVIEWS", "Integridad: Review_ID Único", gx_reviews.expect_column_values_to_be_unique("review_id"))

    render_dashboard()
    spark.stop()

if __name__ == "__main__":
    main()

Dataset,Regla de Negocio,Estado,Filas Erradas,% Error
LISTINGS,Integridad: Listing_ID Único,PASS,0,0.00%
LISTINGS,Consistencia: Precio > 0,FAIL,113,0.04%
LISTINGS,Validez: Taxonomía de Habitación,FAIL,"182,005",65.07%
LISTINGS,Completitud: Metadatos Host,FAIL,165,0.06%
REVIEWS,Integridad: Listing_ID existe (FK),PASS,0,0.00%
REVIEWS,Validez: Formato Fecha ISO,PASS,0,0.00%
REVIEWS,Integridad: Review_ID Único,PASS,0,0.00%
